In [4]:
import os
import re
import json
import faiss
import torch

from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter
from typing import Dict, List, Optional, Tuple

import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from openai import OpenAI


# ============================================================
# Configuration
# ============================================================

KB_FOLDER = "PMART_PUML/knowledge_base"
TEST_FOLDER = "PMART_PUML/test"

INDEX_FOLDER = "PMART_UML/.rag_index"

EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

# BGE recommends an instruction for short query -> passage retrieval.
QUERY_INSTRUCTION = (
    "Represent this sentence for searching relevant passages: "
)

OPENAI_MODEL = "gpt-5.4-mini"

# Dense retrieval first, then structural reranking.
DEFAULT_RETRIEVE_K = 12
DEFAULT_EVIDENCE_K = 5

# Fusion:
# dense semantic similarity is the primary signal;
# UML structure is the secondary signal.
DENSE_WEIGHT = 0.80
STRUCTURE_WEIGHT = 0.20


# ============================================================
# Data model
# ============================================================

@dataclass
class PatternDocument:
    """
    One design-pattern exemplar.

    Important:
        The raw PlantUML is preserved, but retrieval operates on
        a canonical representation rather than raw PlantUML only.
    """

    id: str
    pattern: str
    category: str
    intent: str

    roles: List[str]
    tags: List[str]

    entities: List[Dict]
    operations: List[str]

    # (source, relation_type, target, label)
    relationships: List[Tuple[str, str, str, str]]

    raw: str

    # Text embedded in the vector store.
    retrieval_text: str

    # Text representation of UML structure.
    structural_text: str

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_dict(cls, data):
        return cls(**data)


# ============================================================
# Annotation parsing
# ============================================================

ANNOTATION_RE = re.compile(
    r"""
    ^\s*
    ['#*/]+\s*
    (?:
        DP\s+
    )?
    @?
    (?P<key>
        pattern|
        category|
        intent|
        role|
        tag|
        tags
    )
    \s*:\s*
    (?P<value>.+?)
    \s*$
    """,
    re.IGNORECASE | re.VERBOSE,
)


def parse_annotations(text: str) -> Dict:
    """
    Supports annotations such as:

        ' DP pattern: Observer
        ' DP category: Behavioral
        ' DP intent: One-to-many dependency
        ' DP role: Subject
        ' DP role: Observer
        ' DP tags: notification,event

    Also supports:

        ' @pattern: Observer
        ' @role: Subject
    """

    pattern = None
    category = ""
    intent = ""

    roles = []
    tags = []

    for line in text.splitlines():

        match = ANNOTATION_RE.match(line)

        if not match:
            continue

        key = match.group("key").lower()
        value = match.group("value").strip()

        if key == "pattern":
            pattern = value

        elif key == "category":
            category = value

        elif key == "intent":
            intent = value

        elif key == "role":
            roles.append(value)

        elif key in {"tag", "tags"}:
            tags.extend(
                x.strip()
                for x in re.split(r"[,;]", value)
                if x.strip()
            )

    return {
        "pattern": pattern,
        "category": category,
        "intent": intent,
        "roles": sorted(set(roles)),
        "tags": sorted(set(tags)),
    }


# ============================================================
# Ground truth protection
# ============================================================

GROUND_TRUTH_ANNOTATION_RE = re.compile(
    r"""
    ^\s*
    ['#*/]+\s*
    (?:
        DP\s+
    )?
    @?
    (?:
        pattern|
        category|
        intent|
        tag|
        tags
    )
    \s*:
    """,
    re.IGNORECASE | re.VERBOSE,
)


def remove_ground_truth_annotations(text: str) -> str:
    """
    Prevent evaluation leakage.

    The query must NOT contain:
        DP pattern:
        DP category:
        DP intent:
        DP tags:

    DP roles are kept because they can represent structural roles
    rather than directly naming the target pattern.
    """

    lines = []

    for line in text.splitlines():

        if GROUND_TRUTH_ANNOTATION_RE.match(line):
            continue

        lines.append(line)

    return "\n".join(lines)


# ============================================================
# UML parsing
# ============================================================

ENTITY_RE = re.compile(
    r"""
    ^\s*
    (?:
        abstract\s+
    )?
    (?P<kind>
        class|
        interface|
        component|
        actor|
        object|
        enum
    )
    \s+
    (?P<name>
        [A-Za-z_][A-Za-z0-9_.$:]
    *)
    """,
    re.IGNORECASE | re.VERBOSE,
)


OPERATION_RE = re.compile(
    r"""
    ^\s*
    [+\-#~]
    \s*
    (?P<name>[A-Za-z_][A-Za-z0-9_]*)
    \s*\(
    """,
    re.VERBOSE,
)


RELATION_RE = re.compile(
    r"""
    ^\s*
    (?P<left>[A-Za-z_][A-Za-z0-9_.$:] *)
    \s*
    (?P<arrow>
        <\|--|
        --\|>|
        <\|\.\.|
        \.\.\|>|
        \*--|
        --\*|
        o--|
        --o|
        -->|
        <--
        |\.\.>|
        <\.\.|
        --|
        \.\.
    )
    \s*
    (?P<right>[A-Za-z_][A-Za-z0-9_.$:] *)
    (?:\s*:\s*(?P<label>.*))?
    \s*$
    """,
    re.IGNORECASE | re.VERBOSE,
)


def clean_name(name: str) -> str:

    name = name.strip().strip('"')

    # PlantUML may use namespaces.
    # Preserve the final semantic class name.
    if "::" in name:
        name = name.split("::")[-1]

    if "." in name:
        name = name.split(".")[-1]

    return name.strip()


def relation_type(arrow: str) -> str:
    """
    Normalize PlantUML arrows into semantic UML relationship types.
    """

    if arrow in {"<|--", "--|>"}:
        return "inheritance"

    if arrow in {"<|..", "..|>"}:
        return "implementation"

    if arrow in {"*--", "--*"}:
        return "composition"

    if arrow in {"o--", "--o"}:
        return "aggregation"

    if arrow in {"..>", "<.."}:
        return "dependency"

    if arrow in {"-->", "<--"}:
        return "association"

    if arrow in {"--", ".."}:
        return "association"

    return "unknown"


def parse_uml_structure(text: str):
    """
    Extract:

        - class/interface/component declarations
        - operations
        - typed relationships

    We deliberately do NOT attempt to render PlantUML.
    For pattern detection, semantic structure matters more than
    the rendered image.
    """

    entities = []
    operations = []
    relationships = []

    for line in text.splitlines():

        line = line.strip()

        if not line:
            continue

        # ----------------------------
        # Entities
        # ----------------------------

        entity_match = ENTITY_RE.match(line)

        if entity_match:

            kind = entity_match.group("kind").lower()
            name = clean_name(entity_match.group("name"))

            entities.append({
                "name": name,
                "type": kind,
            })

        # ----------------------------
        # Operations
        # ----------------------------

        operation_match = OPERATION_RE.match(line)

        if operation_match:

            operations.append(
                operation_match.group("name").lower()
            )

        # ----------------------------
        # Relationships
        # ----------------------------

        relation_match = RELATION_RE.match(line)

        if relation_match:

            left = clean_name(
                relation_match.group("left")
            )

            right = clean_name(
                relation_match.group("right")
            )

            arrow = relation_match.group("arrow")

            label = (
                relation_match.group("label") or ""
            ).strip()

            relationships.append(
                (
                    left,
                    relation_type(arrow),
                    right,
                    label,
                )
            )

    # Deduplicate.
    entities = list(
        {
            (x["name"], x["type"]): x
            for x in entities
        }.values()
    )

    operations = sorted(set(operations))

    relationships = sorted(
        set(relationships)
    )

    return {
        "entities": entities,
        "operations": operations,
        "relationships": relationships,
    }


# ============================================================
# Canonical representation
# ============================================================

def structural_representation(
    uml: Dict,
) -> str:

    entity_lines = []

    for entity in sorted(
        uml["entities"],
        key=lambda x: x["name"].lower(),
    ):
        entity_lines.append(
            f"{entity['type']} {entity['name']}"
        )

    relation_lines = []

    for left, rel, right, label in uml["relationships"]:

        if label:
            relation_lines.append(
                f"{left} -[{rel}]-> {right} "
                f"label={label}"
            )
        else:
            relation_lines.append(
                f"{left} -[{rel}]-> {right}"
            )

    operation_text = ", ".join(
        uml["operations"]
    )

    return "\n".join([
        "ENTITIES:",
        *entity_lines,
        "",
        "OPERATIONS:",
        operation_text,
        "",
        "RELATIONSHIPS:",
        *relation_lines,
    ])


def build_retrieval_text(
    annotations: Dict,
    uml: Dict,
    include_pattern: bool,
) -> str:

    sections = []

    if include_pattern and annotations["pattern"]:
        sections.append(
            f"KNOWN DESIGN PATTERN: "
            f"{annotations['pattern']}"
        )

    if include_pattern and annotations["category"]:
        sections.append(
            f"CATEGORY: "
            f"{annotations['category']}"
        )

    if annotations["intent"] and include_pattern:
        sections.append(
            f"INTENT: "
            f"{annotations['intent']}"
        )

    if annotations["roles"]:
        sections.append(
            "PATTERN ROLES: "
            + ", ".join(annotations["roles"])
        )

    if annotations["tags"] and include_pattern:
        sections.append(
            "TAGS: "
            + ", ".join(annotations["tags"])
        )

    sections.append(
        "UML STRUCTURE:\n"
        + structural_representation(uml)
    )

    return "\n\n".join(sections)


# ============================================================
# Pattern name fallback
# ============================================================

def infer_pattern_from_filename(
    filename: str,
) -> str:

    stem = Path(filename).stem

    # Recommended naming convention:
    #
    # Observer_001.puml
    # Observer-001.puml
    #
    # Only strip a final numerical exemplar ID.
    match = re.match(
        r"^(.*?)[_-]\d+$",
        stem,
    )

    if match:
        return match.group(1).strip()

    return stem


# ============================================================
# Document creation
# ============================================================

def parse_document(
    path: str | Path,
    remove_query_ground_truth: bool = False,
) -> PatternDocument:

    path = Path(path)

    raw = path.read_text(
        encoding="utf-8"
    )

    annotations = parse_annotations(raw)

    uml_source = raw

    if remove_query_ground_truth:
        uml_source = remove_ground_truth_annotations(
            raw
        )

    uml = parse_uml_structure(uml_source)

    pattern = annotations["pattern"]

    if not pattern:
        pattern = infer_pattern_from_filename(
            path.name
        )

    structural_text = structural_representation(
        uml
    )

    retrieval_text = build_retrieval_text(
        annotations=annotations,
        uml=uml,
        include_pattern=not remove_query_ground_truth,
    )

    return PatternDocument(
        id=path.name,
        pattern=pattern,
        category=annotations["category"],
        intent=annotations["intent"],
        roles=annotations["roles"],
        tags=annotations["tags"],
        entities=uml["entities"],
        operations=uml["operations"],
        relationships=uml["relationships"],
        raw=raw,
        retrieval_text=retrieval_text,
        structural_text=structural_text,
    )


def load_documents(
    folder_path: str | Path,
    remove_query_ground_truth: bool = False,
) -> List[PatternDocument]:

    folder_path = Path(folder_path)

    if not folder_path.exists():
        raise FileNotFoundError(
            folder_path
        )

    files = sorted(
        list(folder_path.rglob("*.puml"))
        + list(folder_path.rglob("*.plantuml"))
    )

    documents = []

    for path in files:

        try:

            doc = parse_document(
                path,
                remove_query_ground_truth=
                    remove_query_ground_truth,
            )

            documents.append(doc)

        except Exception as exc:

            print(
                f"[WARN] Failed to parse "
                f"{path}: {exc}"
            )

    return documents


# ============================================================
# Embeddings
# ============================================================

class EmbeddingModel:

    def __init__(
        self,
        model_name: str = EMBEDDING_MODEL,
    ):

        self.device = (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

        self.model = SentenceTransformer(
            model_name,
            device=self.device,
        )

    def encode_documents(
        self,
        texts: List[str],
        batch_size: int = 16,
    ) -> np.ndarray:

        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )

        return np.asarray(
            embeddings,
            dtype=np.float32,
        )

    def encode_query(
        self,
        text: str,
    ) -> np.ndarray:

        query = QUERY_INSTRUCTION + text

        embedding = self.model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        return np.asarray(
            embedding,
            dtype=np.float32,
        )


# ============================================================
# FAISS vector store
# ============================================================

class VectorStore:

    def __init__(self):

        self.index = None
        self.documents: List[PatternDocument] = []

    def build(
        self,
        documents: List[PatternDocument],
        embeddings: np.ndarray,
    ):

        if len(documents) == 0:
            raise ValueError(
                "No documents found."
            )

        if len(documents) != len(embeddings):
            raise ValueError(
                "Documents and embeddings mismatch."
            )

        dimension = embeddings.shape[1]

        # Because embeddings are L2-normalized:
        # inner product == cosine similarity.
        self.index = faiss.IndexFlatIP(
            dimension
        )

        self.index.add(
            embeddings.astype(np.float32)
        )

        self.documents = documents

    def search(
        self,
        query_embedding: np.ndarray,
        top_k: int,
    ):

        if self.index is None:
            raise RuntimeError(
                "FAISS index is empty."
            )

        top_k = min(
            top_k,
            len(self.documents),
        )

        scores, indices = self.index.search(
            query_embedding,
            top_k,
        )

        results = []

        for score, idx in zip(
            scores[0],
            indices[0],
        ):

            if idx < 0:
                continue

            results.append({
                "document": self.documents[idx],
                "dense_score": float(score),
            })

        return results

    def save(
        self,
        index_folder: str | Path,
    ):

        index_folder = Path(index_folder)

        index_folder.mkdir(
            parents=True,
            exist_ok=True,
        )

        faiss.write_index(
            self.index,
            str(
                index_folder / "patterns.faiss"
            ),
        )

        metadata = [
            doc.to_dict()
            for doc in self.documents
        ]

        (
            index_folder / "metadata.json"
        ).write_text(
            json.dumps(
                metadata,
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

    @classmethod
    def load(
        cls,
        index_folder: str | Path,
    ):

        index_folder = Path(index_folder)

        store = cls()

        store.index = faiss.read_index(
            str(
                index_folder / "patterns.faiss"
            )
        )

        metadata = json.loads(
            (
                index_folder / "metadata.json"
            ).read_text(
                encoding="utf-8"
            )
        )

        store.documents = [
            PatternDocument.from_dict(x)
            for x in metadata
        ]

        return store


# ============================================================
# Structural similarity
# ============================================================

def multiset_jaccard(
    a: List[str],
    b: List[str],
) -> float:

    ca = Counter(a)
    cb = Counter(b)

    keys = set(ca) | set(cb)

    if not keys:
        return 1.0

    minimum = sum(
        min(ca[k], cb[k])
        for k in keys
    )

    maximum = sum(
        max(ca[k], cb[k])
        for k in keys
    )

    if maximum == 0:
        return 1.0

    return minimum / maximum


def entity_type_similarity(
    query: PatternDocument,
    candidate: PatternDocument,
) -> float:

    q_types = [
        x["type"]
        for x in query.entities
    ]

    c_types = [
        x["type"]
        for x in candidate.entities
    ]

    return multiset_jaccard(
        q_types,
        c_types,
    )


def relationship_similarity(
    query: PatternDocument,
    candidate: PatternDocument,
) -> float:

    q_relations = [
        x[1]
        for x in query.relationships
    ]

    c_relations = [
        x[1]
        for x in candidate.relationships
    ]

    return multiset_jaccard(
        q_relations,
        c_relations,
    )


def role_similarity(
    query: PatternDocument,
    candidate: PatternDocument,
) -> Optional[float]:

    if not query.roles:
        return None

    q = set(
        r.lower()
        for r in query.roles
    )

    c = set(
        r.lower()
        for r in candidate.roles
    )

    if not q or not c:
        return 0.0

    return len(q & c) / len(q | c)


def structural_similarity(
    query: PatternDocument,
    candidate: PatternDocument,
) -> float:

    entity_score = entity_type_similarity(
        query,
        candidate,
    )

    relation_score = relationship_similarity(
        query,
        candidate,
    )

    role_score = role_similarity(
        query,
        candidate,
    )

    # Base structural score.
    scores = [
        (0.45, entity_score),
        (0.55, relation_score),
    ]

    # Only use role annotations when they exist
    # in the query.
    if role_score is not None:
        scores.append(
            (0.20, role_score)
        )

        # Renormalize weights.
        total_weight = sum(
            weight
            for weight, _ in scores
        )

        return sum(
            weight * score
            for weight, score in scores
        ) / total_weight

    return sum(
        weight * score
        for weight, score in scores
    )


# ============================================================
# RAG retrieval + reranking
# ============================================================

class PatternRAG:

    def __init__(
        self,
        embedding_model: str = EMBEDDING_MODEL,
    ):

        self.embedder = EmbeddingModel(
            embedding_model
        )

        self.vector_store = VectorStore()

    # --------------------------------------------------------
    # Build
    # --------------------------------------------------------

    def build(
        self,
        kb_folder: str | Path,
        save_folder: Optional[str | Path] = None,
    ):

        print(
            "Loading knowledge base..."
        )

        documents = load_documents(
            kb_folder
        )

        if not documents:
            raise ValueError(
                "Knowledge base is empty."
            )

        print(
            f"Loaded {len(documents)} "
            f"exemplars."
        )

        texts = [
            doc.retrieval_text
            for doc in documents
        ]

        print(
            "Creating document embeddings..."
        )

        embeddings = (
            self.embedder.encode_documents(
                texts
            )
        )

        print(
            "Building FAISS index..."
        )

        self.vector_store.build(
            documents,
            embeddings,
        )

        if save_folder:
            self.vector_store.save(
                save_folder
            )

        return self

    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    def load(
        self,
        index_folder: str | Path,
    ):

        self.vector_store = (
            VectorStore.load(
                index_folder
            )
        )

        return self

    # --------------------------------------------------------
    # Retrieve
    # --------------------------------------------------------

    def retrieve(
        self,
        plantuml_text: str,
        retrieve_k: int = DEFAULT_RETRIEVE_K,
        evidence_k: int = DEFAULT_EVIDENCE_K,
    ):

        # IMPORTANT:
        # remove ground-truth annotations from the QUERY.
        query_doc = parse_document_from_text(
            plantuml_text
        )

        query_embedding = (
            self.embedder.encode_query(
                query_doc.retrieval_text
            )
        )

        dense_results = (
            self.vector_store.search(
                query_embedding,
                retrieve_k,
            )
        )

        # ----------------------------
        # Structural reranking
        # ----------------------------

        reranked = []

        for result in dense_results:

            doc = result["document"]

            dense_score = (
                result["dense_score"]
            )

            structure_score = (
                structural_similarity(
                    query_doc,
                    doc,
                )
            )

            final_score = (
                DENSE_WEIGHT * dense_score
                + STRUCTURE_WEIGHT *
                    structure_score
            )

            reranked.append({
                "document": doc,
                "dense_score": dense_score,
                "structure_score":
                    structure_score,
                "final_score": final_score,
            })

        reranked.sort(
            key=lambda x: x["final_score"],
            reverse=True,
        )

        # ----------------------------------------------------
        # Pattern-level aggregation
        #
        # Use MAX rather than SUM.
        #
        # Otherwise a pattern with many exemplars gets an
        # artificial advantage.
        # ----------------------------------------------------

        pattern_best = {}

        for result in reranked:

            pattern = result[
                "document"
            ].pattern

            if (
                pattern not in pattern_best
                or result["final_score"]
                > pattern_best[pattern][
                    "final_score"
                ]
            ):
                pattern_best[pattern] = result

        pattern_ranking = sorted(
            pattern_best.values(),
            key=lambda x: x["final_score"],
            reverse=True,
        )

        return {
            "query": query_doc,
            "exemplars": reranked[
                :evidence_k
            ],
            "pattern_ranking": pattern_ranking,
        }


# ============================================================
# Parse query without using its ground-truth label
# ============================================================

def parse_document_from_text(
    raw: str,
) -> PatternDocument:

    # Parse metadata for roles, but do not allow target
    # pattern/category/intent/tags into retrieval.
    annotations = parse_annotations(
        raw
    )

    query_raw = remove_ground_truth_annotations(
        raw
    )

    uml = parse_uml_structure(
        query_raw
    )

    structural_text = structural_representation(
        uml
    )

    # Query representation intentionally excludes
    # the target label.
    retrieval_text = build_retrieval_text(
        annotations=annotations,
        uml=uml,
        include_pattern=False,
    )

    return PatternDocument(
        id="query",
        pattern="",
        category="",
        intent="",
        roles=annotations["roles"],
        tags=[],
        entities=uml["entities"],
        operations=uml["operations"],
        relationships=uml["relationships"],
        raw=raw,
        retrieval_text=retrieval_text,
        structural_text=structural_text,
    )


# ============================================================
# Candidate ranking
# ============================================================

def retrieval_prediction(
    pattern_ranking: List[Dict],
) -> str:

    if not pattern_ranking:
        return "Unknown"

    return (
        pattern_ranking[0]
        ["document"]
        .pattern
    )


# ============================================================
# RAG generation
# ============================================================

class PatternReasoner:

    def __init__(
        self,
        model: str = OPENAI_MODEL,
    ):

        self.model = model

        api_key = os.getenv(
            "OPENAI_API_KEY"
        )

        self.client = (
            OpenAI(api_key=api_key)
            if api_key
            else None
        )

    def reason(
        self,
        query: PatternDocument,
        retrieved: List[Dict],
    ) -> Dict:

        if self.client is None:
            raise RuntimeError(
                "OPENAI_API_KEY is not set."
            )

        evidence_blocks = []

        candidate_names = []

        for rank, result in enumerate(
            retrieved,
            start=1,
        ):

            doc = result["document"]

            candidate_names.append(
                doc.pattern
            )

            evidence_blocks.append(
                f"""
EVIDENCE {rank}
Pattern: {doc.pattern}
Category: {doc.category}
Intent: {doc.intent}
Roles: {", ".join(doc.roles)}
Dense similarity: {result["dense_score"]:.4f}
Structural similarity: {result["structure_score"]:.4f}
Final retrieval score: {result["final_score"]:.4f}

STRUCTURE:
{doc.structural_text}
""".strip()
            )

        evidence = "\n\n".join(
            evidence_blocks
        )

        query_without_gt = (
            remove_ground_truth_annotations(
                query.raw
            )
        )

        prompt = f"""
You are a software architecture expert
specialized in GoF design-pattern detection.

Your task is to identify the most likely GoF
design pattern in the QUERY UML diagram.

IMPORTANT RULES:

1. The retrieved examples are evidence, not truth.
2. Prefer UML structure and relationships over class names.
3. Do not use any hidden ground-truth annotation
   from the query.
4. Select ONLY one pattern from the candidate list.
5. If the evidence is insufficient, return "Unknown".
6. Do not invent a pattern outside the candidates.

CANDIDATES:
{", ".join(candidate_names)}

QUERY:
{query_without_gt}

RETRIEVED EVIDENCE:
{evidence}

Return exactly valid JSON:

{{
  "pattern": "<candidate or Unknown>",
  "confidence": 0.0,
  "evidence": [
    "<structural reason 1>",
    "<structural reason 2>"
  ]
}}
"""

        # Responses API is preferred for current reasoning workflows.
        response = self.client.responses.create(
            model=self.model,
            reasoning={
                "effort": "low"
            },
            input=prompt,
        )

        raw_output = (
            response.output_text.strip()
        )

        try:

            parsed = json.loads(
                raw_output
            )

            predicted = (
                parsed.get(
                    "pattern",
                    "Unknown",
                )
            )

            # Protect against LLM inventing a pattern.
            allowed = {
                x.lower(): x
                for x in candidate_names
            }

            if (
                predicted.lower()
                in allowed
            ):
                parsed["pattern"] = (
                    allowed[
                        predicted.lower()
                    ]
                )

            else:
                parsed["pattern"] = (
                    "Unknown"
                )

            return parsed

        except json.JSONDecodeError:

            # Defensive fallback if the model did not
            # follow JSON exactly.
            text = raw_output.lower()

            for candidate in candidate_names:

                if candidate.lower() in text:
                    return {
                        "pattern": candidate,
                        "confidence": 0.0,
                        "evidence": [
                            raw_output
                        ],
                    }

            return {
                "pattern": "Unknown",
                "confidence": 0.0,
                "evidence": [
                    raw_output
                ],
            }


# ============================================================
# Full detection
# ============================================================

def detect_pattern(
    rag: PatternRAG,
    plantuml_text: str,
    use_llm: bool = True,
    retrieve_k: int = DEFAULT_RETRIEVE_K,
    evidence_k: int = DEFAULT_EVIDENCE_K,
):

    retrieval = rag.retrieve(
        plantuml_text,
        retrieve_k=retrieve_k,
        evidence_k=evidence_k,
    )

    ranking = retrieval[
        "pattern_ranking"
    ]

    dense_prediction = (
        retrieval_prediction(
            ranking
        )
    )

    result = {
        "retrieval_prediction":
            dense_prediction,

        "pattern_ranking": [
            {
                "pattern":
                    r["document"].pattern,
                "score":
                    r["final_score"],
                "dense_score":
                    r["dense_score"],
                "structure_score":
                    r["structure_score"],
                "source":
                    r["document"].id,
            }
            for r in ranking
        ],

        "evidence": [
            {
                "pattern":
                    r["document"].pattern,
                "score":
                    r["final_score"],
                "source":
                    r["document"].id,
                "intent":
                    r["document"].intent,
                "structure":
                    r["document"].structural_text,
            }
            for r in retrieval[
                "exemplars"
            ]
        ],
    }

    # --------------------------------------------------------
    # LLM is a GENERATION/REASONING layer.
    #
    # It is not used to measure retrieval accuracy.
    # --------------------------------------------------------

    if use_llm:

        reasoner = PatternReasoner()

        llm_result = reasoner.reason(
            retrieval["query"],
            retrieval["exemplars"],
        )

        result["llm_prediction"] = (
            llm_result["pattern"]
        )

        result["llm_result"] = (
            llm_result
        )

    else:

        result["llm_prediction"] = None
        result["llm_result"] = None

    return result


# ============================================================
# Evaluation
# ============================================================

def evaluate_retrieval(
    rag: PatternRAG,
    test_folder: str | Path,
    top_ks=(1, 3, 5),
):

    # We keep the original annotations here ONLY
    # to obtain the true label.
    test_docs = load_documents(
        test_folder,
        remove_query_ground_truth=False,
    )

    if not test_docs:
        raise ValueError(
            "No test documents found."
        )

    correct = Counter()
    total = Counter()

    recall_counts = {
        k: 0
        for k in top_ks
    }

    for doc in tqdm(
        test_docs,
        desc="Evaluating retrieval",
    ):

        true_label = doc.pattern

        total[true_label] += 1

        retrieval = rag.retrieve(
            doc.raw,
            retrieve_k=max(top_ks),
            evidence_k=max(top_ks),
        )

        ranked_patterns = [
            x["document"].pattern
            for x in retrieval[
                "pattern_ranking"
            ]
        ]

        # Top-k retrieval recall.
        for k in top_ks:

            if true_label.lower() in {
                x.lower()
                for x in ranked_patterns[:k]
            }:
                recall_counts[k] += 1

        predicted = (
            ranked_patterns[0]
            if ranked_patterns
            else "Unknown"
        )

        if predicted.lower() == (
            true_label.lower()
        ):
            correct[true_label] += 1

        print(
            f"{doc.id} | "
            f"True={true_label} | "
            f"Pred={predicted}"
        )

    total_examples = len(
        test_docs
    )

    top1_accuracy = (
        sum(correct.values())
        / total_examples
    )

    recall_at_k = {
        k: recall_counts[k]
        / total_examples
        for k in top_ks
    }

    per_pattern_accuracy = {
        label:
            correct[label]
            / total[label]
        for label in total
    }

    print(
        "\n=============================="
    )

    print(
        f"Top-1 Accuracy: "
        f"{top1_accuracy:.4f}"
    )

    for k, score in recall_at_k.items():
        print(
            f"Recall@{k}: {score:.4f}"
        )

    print(
        "\nPer-pattern Accuracy:"
    )

    for pattern, score in (
        per_pattern_accuracy.items()
    ):
        print(
            f"{pattern}: {score:.4f}"
        )

    return {
        "top1_accuracy": top1_accuracy,
        "recall_at_k": recall_at_k,
        "per_pattern_accuracy":
            per_pattern_accuracy,
    }


# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # 1. Build KB
    # --------------------------------------------------------

    rag = PatternRAG()

    rag.build(
        KB_FOLDER,
        save_folder=INDEX_FOLDER,
    )

    # --------------------------------------------------------
    # 2. Evaluate retrieval WITHOUT LLM
    #
    # This is the important experiment.
    # --------------------------------------------------------

    evaluate_retrieval(
        rag,
        TEST_FOLDER,
        top_ks=(1, 3, 5),
    )

    # --------------------------------------------------------
    # 3. Example inference
    # --------------------------------------------------------

    query_file = (
        "PLANTU_UML/test/example.puml"
    )

    if Path(query_file).exists():

        query_text = Path(
            query_file
        ).read_text(
            encoding="utf-8"
        )

        result = detect_pattern(
            rag,
            query_text,
            use_llm=True,
        )

        print(
            "\n=============================="
        )

        print(
            "Dense/Reranker prediction:",
            result[
                "retrieval_prediction"
            ],
        )

        print(
            "LLM/RAG prediction:",
            result[
                "llm_prediction"
            ],
        )

        print(
            "\nPattern ranking:"
        )

        for item in result[
            "pattern_ranking"
        ][:5]:

            print(
                f"{item['pattern']:25s} "
                f"final={item['score']:.4f} "
                f"dense={item['dense_score']:.4f} "
                f"struct={item['structure_score']:.4f}"
            )

Loading knowledge base...
Loaded 145 exemplars.
Creating document embeddings...


Batches: 100%|██████████| 10/10 [00:02<00:00,  4.71it/s]


Building FAISS index...


Evaluating retrieval:   0%|          | 0/83 [00:00<?, ?it/s]

Abstract Factory-39.puml | True=Abstract Factory | Pred=Abstract Factory
Abstract Factory-40.puml | True=Abstract Factory | Pred=Abstract Factory
Abstract Factory-41.puml | True=Abstract Factory | Pred=Abstract Factory


...

State-2050.puml | True=State | Pred=State
State-2079.puml | True=State | Pred=Builder-2a
State-2085.puml | True=State | Pred=Template Method-107y


Evaluating retrieval:  93%|█████████▎| 77/83 [00:01<00:00, 87.36it/s] 

Strategy-2070.puml | True=Strategy | Pred=Template Method-109y
Strategy-2071.puml | True=Strategy | Pred=Builder-2a
Strategy-414.puml | True=Strategy | Pred=FactoryMethod-77y
Strategy-427.puml | True=Strategy | Pred=Builder-2a
Template Method-2080.puml | True=Template Method | Pred=Null Object
Template Method-2090.puml | True=Template Method | Pred=Null Object
Template Method-2100.puml | True=Template Method | Pred=FactoryMethod-77y
Template Method-2110.puml | True=Template Method | Pred=State
Template Method-2222.puml | True=Template Method | Pred=State
Template Method-2223.puml | True=Template Method | Pred=Template Method-109y
Template Method-2224.puml | True=Template Method | Pred=Template Method-107y
Template Method-2225.puml | True=Template Method | Pred=Adapter
Template Method-2226.puml | True=Template Method | Pred=FactoryMethod-77y


Evaluating retrieval: 100%|██████████| 83/83 [00:01<00:00, 69.83it/s]

Visitor-37.puml | True=Visitor | Pred=Visitor
Visitor-38.puml | True=Visitor | Pred=Visitor

Top-1 Accuracy: 0.3494
Recall@1: 0.3494
Recall@3: 0.3976
Recall@5: 0.4096

Per-pattern Accuracy:
Abstract Factory: 1.0000
Adapter: 0.6429
Command: 0.2000
Factory Method: 0.0000
Iterator: 0.5714
Proxy: 0.0000
Singleton: 0.0000
State: 0.3333
Strategy: 0.0000
Template Method: 0.0000
Visitor: 1.0000


In [5]:
##############################################
# 3. Vector store with exclusion (leave-one-out)
##############################################
class VectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatIP(dim)
        self.metadata = []

    def add_embeddings(self, embeddings, metadatas):
        self.index.add(np.asarray(embeddings, dtype="float32"))
        self.metadata.extend(metadatas)

    def search(self, q, top_k=5, exclude_ids=()):
        k = min(top_k + len(exclude_ids), self.index.ntotal)
        scores, idxs = self.index.search(q, k)
        out = []
        for s, i in zip(scores[0], idxs[0]):
            if i == -1 or self.metadata[i]["id"] in exclude_ids:
                continue
            out.append({**self.metadata[i], "score": float(s)})
            if len(out) == top_k:
                break
        return out

##############################################
# 4. LLM reasoning: explicit labels + constrained output
##############################################
def llm_reasoning(query_raw, retrieved_docs):
    examples = "\n\n".join(
        f"Example {i+1} (pattern: {d['label']}):\n{d['raw']}"
        for i, d in enumerate(retrieved_docs)
    )
    prompt = (
        "Identify the GoF design pattern implemented in the QUERY diagram.\n\n"
        f"QUERY DIAGRAM:\n{query_raw}\n\n"
        f"ANNOTATED EXAMPLES (retrieved from the knowledge base):\n{examples}"
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are an expert in UML and GoF design patterns."},
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_schema", "json_schema": {
            "name": "pattern_prediction", "strict": True,
            "schema": {"type": "object",
                       "properties": {"pattern": {"type": "string", "enum": GOF_PATTERNS}},
                       "required": ["pattern"], "additionalProperties": False}}},
    )
    import json
    return normalize_prediction(json.loads(resp.choices[0].message.content)["pattern"])

##############################################
# 5. Detection: both predictions in one pass
##############################################
def detect_pattern(query_raw, vector_store, embedder, top_k=5, exclude_ids=(), use_llm=True):
    query_raw = strip_annotations(query_raw)
    q = embedder.encode([semantic_puml_processor(query_raw)])
    results = vector_store.search(q, top_k, exclude_ids)

    votes = defaultdict(float)
    for r in results:
        votes[r["label"]] += r["score"]
    retrieval_pred = max(votes, key=votes.get) if votes else "Unknown"

    return {
        "retrieval_pred": retrieval_pred,
        "llm_pred": llm_reasoning(query_raw, results) if use_llm else None,
        "retrieved": results,
    }

##############################################
# 6. Evaluation
##############################################
def evaluate_detection(test_folder, vector_store, embedder, top_k=5,
                       use_llm=True, leave_one_out=False):
    test_docs = load_and_process_docs(test_folder)
    y_true, y_ret, y_llm, hits = [], [], [], 0

    for doc in tqdm(test_docs):
        out = detect_pattern(doc["raw"], vector_store, embedder, top_k,
                             exclude_ids={doc["id"]} if leave_one_out else (),
                             use_llm=use_llm)
        y_true.append(doc["label"])
        y_ret.append(out["retrieval_pred"])
        y_llm.append(out["llm_pred"])
        hits += any(r["label"] == doc["label"] for r in out["retrieved"])  # hit@k

    report = {"hit@k": hits / len(test_docs)}
    print(f"Retrieval hit@{top_k}: {report['hit@k']:.3f}")
    print("\n== Retrieval only (kNN vote) ==")
    print(classification_report(y_true, y_ret, zero_division=0))
    if use_llm:
        print("== RAG + LLM ==")
        print(classification_report(y_true, y_llm, zero_division=0))
        report["confusion"] = confusion_matrix(y_true, y_llm, labels=GOF_PATTERNS)
    return report